In [1]:
import joblib
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import (
                                    StratifiedKFold, 
                                    GridSearchCV
                                    )
from sklearn.metrics import confusion_matrix
warnings.filterwarnings('ignore')

### 1. Load the data

In [2]:
from pathlib import Path

project_root = Path.cwd()
for parent in [project_root, *project_root.parents]:
    if (parent / 'artifacts').exists():
        project_root = parent
        break

with np.load(project_root / 'artifacts' / 'credit_card_fraud_X_train.npz') as data:
    X_train = data['X_train']
with np.load(project_root / 'artifacts' / 'credit_card_fraud_y_train.npz') as data:
    Y_train = data['y_train']
with np.load(project_root / 'artifacts' / 'credit_card_fraud_X_test.npz') as data:
    X_test = data['X_test']
with np.load(project_root / 'artifacts' / 'credit_card_fraud_y_test.npz') as data:
    Y_test = data['y_test']

### 2. Define Multi Models

In [3]:
lr_param_grid = {
                'max_iter' : [1000, 5000, 10000]
                }
dt_param_grid = {
                'max_depth' : [8, 12, 16, 20],
                'criterion' : ["gini", "entropy", "log_loss"]
                }
rf_param_grid = {
                'n_estimators' : [100],
                'max_depth' : [8, 12],
                'criterion' : ["gini", "entropy", "log_loss"]
                }

param_grids = {
            'Logistic Regression' : lr_param_grid,
            'Decision Tree' : dt_param_grid,
            'Random Forest' : rf_param_grid,
}

### 3. Define Multi Models

In [4]:
models = {
        'Logistic Regression' : LogisticRegression(),
        'Decision Tree' :DecisionTreeClassifier(),
        'Random Forest' : RandomForestClassifier()
        }

### 4. Configure K-Fold CV

In [5]:
cv = StratifiedKFold(
                    n_splits=6,
                    random_state=42,
                    shuffle=True
                    )

#### 5. Multi Model Training

In [6]:
grid_search_results = {}
for model_name, model in models.items():
    print(f"\n--- Tuning {model_name} ---")

    param_grid = param_grids[model_name]

    grid_search = GridSearchCV(
                                estimator=model,
                                param_grid=param_grid,
                                cv=cv, scoring='f1',
                                verbose=1, return_train_score=False
                                )
    
    print(f"Fitting gridSearchCV for {model_name}")

    grid_search.fit(X_train, Y_train)

    grid_search_results[model_name] = grid_search
    
    print(f"{model_name} gridSearchCV completed ...")
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best CV score: {grid_search.best_score_}")


--- Tuning Logistic Regression ---
Fitting gridSearchCV for Logistic Regression
Fitting 6 folds for each of 3 candidates, totalling 18 fits
Logistic Regression gridSearchCV completed ...
Best parameters: {'max_iter': 1000}
Best CV score: 0.7318169755021501

--- Tuning Decision Tree ---
Fitting gridSearchCV for Decision Tree
Fitting 6 folds for each of 12 candidates, totalling 72 fits
Decision Tree gridSearchCV completed ...
Best parameters: {'criterion': 'entropy', 'max_depth': 20}
Best CV score: 0.8361726179971954

--- Tuning Random Forest ---
Fitting gridSearchCV for Random Forest
Fitting 6 folds for each of 6 candidates, totalling 36 fits
Random Forest gridSearchCV completed ...
Best parameters: {'criterion': 'gini', 'max_depth': 12, 'n_estimators': 100}
Best CV score: 0.8691159090199966


In [7]:
grid_search_results

{'Logistic Regression': GridSearchCV(cv=StratifiedKFold(n_splits=6, random_state=42, shuffle=True),
              estimator=LogisticRegression(),
              param_grid={'max_iter': [1000, 5000, 10000]}, scoring='f1',
              verbose=1),
 'Decision Tree': GridSearchCV(cv=StratifiedKFold(n_splits=6, random_state=42, shuffle=True),
              estimator=DecisionTreeClassifier(),
              param_grid={'criterion': ['gini', 'entropy', 'log_loss'],
                          'max_depth': [8, 12, 16, 20]},
              scoring='f1', verbose=1),
 'Random Forest': GridSearchCV(cv=StratifiedKFold(n_splits=6, random_state=42, shuffle=True),
              estimator=RandomForestClassifier(),
              param_grid={'criterion': ['gini', 'entropy', 'log_loss'],
                          'max_depth': [8, 12], 'n_estimators': [100]},
              scoring='f1', verbose=1)}